In [ ]:
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'common.ipynb').is_file()))
get_ipython().run_line_magic('run', '-i "' + str(ROOT / 'src' / 'common.ipynb') + '"')

selected_parameters = select_xgb_parameters()


In [ ]:
import os
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, MaxNLocator
from sklearn.inspection import partial_dependence
from joblib import Parallel, delayed
FIG = OUT / 'figures'
plt.rcdefaults()
plt.rcParams.update({'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42})
PDP_FEATURES = ['HDD', 'CDD', 'LATITUDE', 'LONGITUDE', 'ACS_PCT_HU_ELEC', 'AC_RECS']
N_BOOTSTRAP = 1000
WORKERS = min(8, os.cpu_count() or 1)


In [ ]:
point_rows, inputs, references = ([], {}, {})
for code in SEASONS:
    fitted = fit_global(code)
    data, model = fitted['data'], fitted['model']
    train, test = fitted['train_indices'], fitted['test_indices']
    Xtrain = data.iloc[train][FEATURES].to_numpy(np.float32)
    ytrain = data.iloc[train]['EB_pct'].to_numpy()
    Xtest = data.iloc[test][FEATURES].reset_index(drop=True)
    references[code] = Xtest
    grids, blocks = ([], [])
    for feature in PDP_FEATURES:
        result = partial_dependence(model, Xtest, features=[feature], grid_resolution=50, kind='average', method='brute')
        grid, curve = (result['grid_values'][0], result['average'][0])
        assert len(grid) == 50
        grids.append(grid)
        work = np.tile(Xtest.to_numpy(np.float32), (len(grid), 1))
        work[:, FEATURES.index(feature)] = np.repeat(grid, len(Xtest))
        blocks.append(work)
        for x, y in zip(grid, curve):
            point_rows.append({'model': code, 'feature': feature, 'x': x, 'PDP_pp': y})
    inputs[code] = {'Xtrain': Xtrain, 'ytrain': ytrain, 'evaluation': np.concatenate(blocks), 'grids': grids, 'n_test': len(Xtest)}
points = pd.DataFrame(point_rows)


In [ ]:
def bootstrap_curve(code, replicate):
    data = inputs[code]
    rng = np.random.default_rng(912028 + replicate)
    indices = rng.integers(0, len(data['ytrain']), size=len(data['ytrain']))
    model = xgb.XGBRegressor(**CFG['xgb_params'])
    model.set_params(n_jobs=1)
    model.fit(data['Xtrain'][indices], data['ytrain'][indices])
    prediction = model.get_booster().inplace_predict(data['evaluation'])
    curves = prediction.reshape(len(PDP_FEATURES), 50, data['n_test']).mean(axis=2).astype(np.float32)
    return code, replicate, curves

bootstrap_results = Parallel(n_jobs=WORKERS, prefer='threads', verbose=10)(
    delayed(bootstrap_curve)(code, replicate)
    for replicate in range(N_BOOTSTRAP) for code in SEASONS)


In [ ]:
rows = []
for code in SEASONS:
    samples = np.asarray([curves for season, replicate, curves in bootstrap_results if season == code], dtype=np.float64)
    assert samples.shape == (N_BOOTSTRAP, len(PDP_FEATURES), 50)
    lower, upper = np.quantile(samples, [0.025, 0.975], axis=0)
    for j, feature in enumerate(PDP_FEATURES):
        curve = points.loc[points['model'].eq(code) & points['feature'].eq(feature)].reset_index(drop=True)
        for k, row in curve.iterrows():
            rows.append({**row.to_dict(), 'label': LABELS[feature],
                'CI95_lower_pp': lower[j, k], 'CI95_upper_pp': upper[j, k],
                'bootstrap_mean_pp': samples[:, j, k].mean(),
                'bootstrap_sd_pp': samples[:, j, k].std(ddof=1),
                'n_bootstrap': N_BOOTSTRAP, 'background_n': inputs[code]['n_test']})
summary = pd.DataFrame(rows)
assert len(summary) == 1200
assert summary['n_bootstrap'].eq(1000).all()
assert summary['background_n'].eq(566).all()
assert (summary['CI95_lower_pp'] <= summary['CI95_upper_pp']).all()
assert np.isfinite(summary[['x', 'PDP_pp', 'CI95_lower_pp', 'CI95_upper_pp']].to_numpy()).all()


In [ ]:
COLORS = {'SP': '#338535', 'SU': '#AD352C', 'FA': '#D48104', 'WI': '#297BBD'}
limits = (2.0, 4.0)
ticks = np.arange(2.0, 4.01, 0.5)
rug_y = 2.04
np.random.seed(SEED)
fig, axes = plt.subplots(6, 4, figsize=(20, 27))
for i, feature in enumerate(PDP_FEATURES):
    for j, season in enumerate(SEASONS):
        ax = axes[i, j]
        rows = summary.loc[summary['feature'].eq(feature) & summary['model'].eq(season)]
        assert len(rows) == 50 and rows['x'].is_monotonic_increasing
        ax.set_yticks(ticks)
        ax.set_ylim(*limits)
        ax.set_autoscaley_on(False)
        ax.plot(rows['x'], rows['PDP_pp'], linewidth=2.5, color=COLORS[season])
        ax.fill_between(rows['x'], rows['CI95_lower_pp'], rows['CI95_upper_pp'], alpha=0.15, color=COLORS[season])
        values = references[season][feature].dropna().to_numpy()
        sampled = np.random.choice(values, size=min(1000, len(values)), replace=False)
        ax.scatter(sampled, np.full(len(sampled), rug_y), alpha=0.3, s=1, color=COLORS[season])
        ax.set_xlabel(LABELS[feature], fontsize=10)
        ax.set_ylabel('Monthly Energy Burden (%)' if j == 0 else '', fontsize=10)
        ax.grid(True, alpha=0.3, linestyle='--')
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
        if i == 0:
            ax.set_title(SEASON_NAMES[season], fontsize=14, color=COLORS[season])
        if j == 0:
            ax.text(-0.12, 1.03, chr(97 + i), transform=ax.transAxes, fontsize=18, fontweight='bold', ha='left', va='bottom')
        ax.set_yticks(ticks)
        ax.set_ylim(*limits)
plt.tight_layout()
fig.canvas.draw()
for ax in axes.flat:
    boxes = [t.get_window_extent(fig.canvas.get_renderer()) for t in ax.get_xticklabels() if t.get_visible() and t.get_text()]
    if any((a.x1 + 2 > b.x0 for a, b in zip(boxes, boxes[1:]))):
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
plt.tight_layout()
for ext in ['png', 'pdf', 'svg']:
    fig.savefig(FIG / f'Figure5.{ext}', dpi=300, bbox_inches='tight')
plt.close(fig)
